In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_F.xpt"
csv_file = DATA_DIR / "HOQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 HOQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOQ040",
    "HOD050",
    "HOD060",
    "HOQ065",
    "HOQ070",
    "HOQ080"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HOQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10537:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 7:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,537 rows x 7 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# The attached HOQ_F contains NONE.
#
# IMPORTANT:
#
# If a future copy contains tiny values, do NOT
# automatically convert them to zero without
# validating the variable and CDC codebook.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "were found in HOQ_F. "
        "Do not automatically replace."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifacts."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# Always run this check BEFORE converting fields
# to Int64.
#
# HOQ_F contains no genuine fractional values.
#
# This protection remains important for:
#
#     Dietary values
#     Lab measurements
#     Examination measurements
#     Sample weights
#
# Those decimals must remain exactly as released.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in HOQ_F."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# After proving there are no genuine decimal values,
# every HOQ_F field can safely be converted to Int64.
#
# IMPORTANT:
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All HOQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC CHECK - HOQ040
# ============================================================

#
# When was home built
#
# CDC:
#
# 1  = 1990 to present       2,163
# 2  = 1978 to 1989          1,158
# 3  = 1960 to 1977          1,707
# 4  = 1950 to 1959            852
# 5  = 1940 to 1949            520
# 6  = Before 1940           1,460
# 77 = Refused                  27
# 99 = Don't know            2,582
# Missing                       68
#

EXPECTED_HOQ040 = {
    1: 2163,
    2: 1158,
    3: 1707,
    4: 852,
    5: 520,
    6: 1460,
    77: 27,
    99: 2582
}


actual = (
    df["HOQ040"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- HOQ040 ---")


if (
    actual != EXPECTED_HOQ040
    or
    int(
        df["HOQ040"]
        .isna()
        .sum()
    )
    != 68
):

    raise ValueError(
        "HOQ040 differs from CDC.\n"
        f"Expected: {EXPECTED_HOQ040}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: HOQ040 matches CDC."
)


# ============================================================
# 9. CDC CHECK - HOD050
# ============================================================

#
# Number of rooms in home
#
# CDC:
#
# 1-12  = 10,348
# 13    = 13 or more = 97
# 777   = Refused = 13
# 999   = Don't know = 11
# Missing = 68
#
# CDC notes:
#
# Values >= 13 were grouped and released as 13.
#

s = df["HOD050"]


print("\n--- HOD050 ---")


print(
    "1-12:",
    int(
        s.between(
            1,
            12
        ).sum()
    )
)


print(
    "13 or more:",
    int(
        s.eq(13)
        .sum()
    )
)


print(
    "Refused (777):",
    int(
        s.eq(777)
        .sum()
    )
)


print(
    "Don't know (999):",
    int(
        s.eq(999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            12
        ).sum()
    )
    == 10348
    and
    int(
        s.eq(13)
        .sum()
    )
    == 97
    and
    int(
        s.eq(777)
        .sum()
    )
    == 13
    and
    int(
        s.eq(999)
        .sum()
    )
    == 11
    and
    int(
        s.isna()
        .sum()
    )
    == 68
):

    raise ValueError(
        "HOD050 differs from CDC."
    )


print(
    "PASS: HOD050 matches CDC."
)


# ============================================================
# 10. CDC CHECK - HOD060
# ============================================================

#
# How many years family lived in home
#
# IMPORTANT:
#
# CDC states these data were originally collected as
# continuous values but were collapsed into categories
# for public release.
#
# Therefore these released values are integer codes,
# NOT genuine decimals.
#
# CDC:
#
# 1  = Less than one year    1,590
# 2  = 1-2 years             1,864
# 3  = 3-5 years             2,118
# 4  = 6-10 years            1,938
# 5  = More than 10 years    2,925
# 77 = Refused                  18
# 99 = Don't know               16
# Missing                       68
#

EXPECTED_HOD060 = {
    1: 1590,
    2: 1864,
    3: 2118,
    4: 1938,
    5: 2925,
    77: 18,
    99: 16
}


actual = (
    df["HOD060"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- HOD060 ---")


if (
    actual != EXPECTED_HOD060
    or
    int(
        df["HOD060"]
        .isna()
        .sum()
    )
    != 68
):

    raise ValueError(
        "HOD060 differs from CDC.\n"
        f"Expected: {EXPECTED_HOD060}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: HOD060 matches CDC."
)


# ============================================================
# 11. CDC CHECK - HOQ065
# ============================================================

#
# Home owned, bought, rented, other
#
# CDC:
#
# 1 = Owned / being bought     5,936
# 2 = Rented                    4,281
# 3 = Other arrangement           243
# 7 = Refused                       5
# 9 = Don't know                    4
# Missing                         68
#

EXPECTED_HOQ065 = {
    1: 5936,
    2: 4281,
    3: 243,
    7: 5,
    9: 4
}


actual = (
    df["HOQ065"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- HOQ065 ---")


if (
    actual != EXPECTED_HOQ065
    or
    int(
        df["HOQ065"]
        .isna()
        .sum()
    )
    != 68
):

    raise ValueError(
        "HOQ065 differs from CDC.\n"
        f"Expected: {EXPECTED_HOQ065}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: HOQ065 matches CDC."
)


# ============================================================
# 12. CDC CHECK - HOQ070
# ============================================================

#
# Source of tap water
#
# CDC:
#
# 1 = Private/public water company   8,997
# 2 = Private/public well            1,187
# 3 = Something else                    94
# 7 = Refused                            5
# 9 = Don't know                       186
# Missing                               68
#

EXPECTED_HOQ070 = {
    1: 8997,
    2: 1187,
    3: 94,
    7: 5,
    9: 186
}


actual = (
    df["HOQ070"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- HOQ070 ---")


if (
    actual != EXPECTED_HOQ070
    or
    int(
        df["HOQ070"]
        .isna()
        .sum()
    )
    != 68
):

    raise ValueError(
        "HOQ070 differs from CDC.\n"
        f"Expected: {EXPECTED_HOQ070}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: HOQ070 matches CDC."
)


# ============================================================
# 13. CDC CHECK - HOQ080
# ============================================================

#
# Water treatment devices used or not
#
# CDC:
#
# 1 = Yes          2,734
# 2 = No           7,712
# 7 = Refused          3
# 9 = Don't know      20
# Missing             68
#

EXPECTED_HOQ080 = {
    1: 2734,
    2: 7712,
    7: 3,
    9: 20
}


actual = (
    df["HOQ080"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- HOQ080 ---")


if (
    actual != EXPECTED_HOQ080
    or
    int(
        df["HOQ080"]
        .isna()
        .sum()
    )
    != 68
):

    raise ValueError(
        "HOQ080 differs from CDC.\n"
        f"Expected: {EXPECTED_HOQ080}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: HOQ080 matches CDC."
)


# ============================================================
# 14. COMMON MISSINGNESS CHECK
# ============================================================

#
# All six housing variables have exactly 68 missing
# values in the released file.
#

HOUSING_COLUMNS = [
    "HOQ040",
    "HOD050",
    "HOD060",
    "HOQ065",
    "HOQ070",
    "HOQ080"
]


print("\n--- COMMON MISSINGNESS CHECK ---")


for col in HOUSING_COLUMNS:

    missing = int(
        df[col]
        .isna()
        .sum()
    )


    print(
        f"{col}: {missing} missing"
    )


    if missing != 68:

        raise ValueError(
            f"{col}: expected 68 missing values, "
            f"found {missing}."
        )


print(
    "PASS: Housing-variable missing counts "
    "match CDC."
)


# ============================================================
# 15. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HOQ_F requires NO numeric corrections.
#
# Therefore every numeric value must remain exactly
# equal to the original XPT.
#
# Int64 conversion changes only representation:
#
#     51624.0 -> 51624
#     1.0     -> 1
#     999.0   -> 999
#
# The numeric values themselves do not change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 7 variables preserve "
    "original XPT values exactly."
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO rounding
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 17. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 18. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 7 CSV columns match "
    "source XPT values exactly."
)


# ============================================================
# 19. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 20. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 21. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# HOQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere in
# this particular CSV.
#
# NOTE:
#
# Do NOT use this same global test on Dietary or
# mixed continuous-measurement files.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 22. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 23. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "HOQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 HOQ_F XPT -> CSV
Rows: 10,537
Columns: 7
PASS: Structure = 10,537 rows x 7 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifacts.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in HOQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All HOQ_F fields formatted without unnecessary .0.

--- HOQ040 ---
PASS: HOQ040 matches CDC.

--- HOD050 ---
1-12: 10348
13 or more: 97
Refused (777): 13
Don't know (999): 11
Missing: 68
PASS: HOD050 matches CDC.

--- HOD060 ---
PASS: HOD060 matches CDC.

--- HOQ065 ---
PASS: HOQ065 matches CDC.

--- HOQ070 ---
PASS: HOQ070 matches CDC.

--- HOQ080 ---
PASS: HOQ080 matches CDC.

--- COMMON MISSINGNESS CHECK ---
HOQ040: 68 missing
HOD050: 68 missing
HOD060: 68 missing
HOQ065: 68 missing
HOQ070: 68 missing
HOQ080: 68 missing
PASS: Housing-variable missing counts match CDC.

--- VALUE 